# Phase 10 — Stress Testing & Tail Risk

**Objective:** Evaluate strategy robustness during historical stress periods, model tail risk via EVT (GPD) and copulas, and run Monte Carlo simulations.

**Dependencies:** Phase 9 (backtest results), Phase 1 (master data)

**Outputs:**
- `stress_test_results.csv`, `var_cvar_table.csv`, `evt_parameters.csv`, `backtest_var_results.csv`

In [1]:
# === Setup & Imports ===
import sys, warnings, logging
from datetime import datetime
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import genpareto, norm, t as student_t, chi2

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))

from src.config import (
    PROJECT_ROOT, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR,
    TICKERS, KEY_EVENTS, RANDOM_STATE, MC_PATHS, MC_HORIZON, COLORS
)
from src.validation import validate_parquet, check_nan_propagation
from src.risk_metrics import (
    var_historical, var_gaussian, var_cornish_fisher, var_student_t, var_evt_gpd,
    cvar_historical, portfolio_cvar,
    kupiec_pof_test, christoffersen_independence_test, conditional_coverage_test,
    fit_clayton_copula, to_pseudo_observations,
    compute_all_metrics, mean_excess_function, hill_estimator,
    rolling_var_cvar
)
from src.portfolio_optimization import covariance_ledoit_wolf
from src.garch_utils import ensure_psd
from src.visualization import setup_style, save_fig

setup_style()
np.random.seed(RANDOM_STATE)

PHASE_NUM = 10
logging.basicConfig(
    filename=str(LOGS_DIR / f'phase_{PHASE_NUM}_{datetime.now():%Y%m%d_%H%M}.log'),
    level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)
logger.info("Phase 10 started")
print("Phase 10 — Stress Testing & Tail Risk")

Phase 10 — Stress Testing & Tail Risk


## 10.1 Load Data

In [2]:
# Load backtest results (factor timing)
backtest_returns = pd.read_parquet(PROCESSED_DIR / 'backtest_returns.parquet')
backtest_nav = pd.read_parquet(PROCESSED_DIR / 'backtest_nav.parquet')
print(f"Backtest returns: {backtest_returns.shape}")
print(f"Strategies: {list(backtest_returns.columns)}")

# Load tech portfolio master data
master_data = pd.read_parquet(PROCESSED_DIR / 'master_data.parquet')
tech_tickers = [t for t in TICKERS if t in master_data.columns]
# Filter out tickers that are entirely NaN (e.g. XYZ placeholder)
tech_tickers = [t for t in tech_tickers if master_data[t].notna().sum() > 100]
tech_returns = master_data[tech_tickers].pct_change()  # NaN handled per-ticker downstream
print(f"\nTech returns: {tech_returns.shape}, tickers: {len(tech_tickers)}")

# Load factor returns
factor_returns = pd.read_parquet(PROCESSED_DIR / 'factor_returns_full.parquet')
print(f"Factor returns: {factor_returns.shape}")

Backtest returns: (64, 6)
Strategies: ['BL Dynamic', 'CVaR Dynamic', 'Equal-Weight', 'Inverse-Vol', 'Market', '60/40']

Tech returns: (2556, 19), tickers: 19
Factor returns: (232, 4)


## 10.2 Named Stress Period Analysis

In [3]:
# Define stress periods
STRESS_PERIODS = {
    'GFC': ('2007-10-01', '2009-03-31'),
    'European Debt': ('2011-07-01', '2011-12-31'),
    'Taper Tantrum': ('2013-05-01', '2013-09-30'),
    'China/Oil': ('2015-08-01', '2016-02-29'),
    'COVID-19': ('2020-02-01', '2020-04-30'),
    'Rate Shock 2022': ('2022-01-01', '2022-10-31'),
}

stress_results = []
strategies = backtest_returns.columns.tolist()

for event_name, (start, end) in STRESS_PERIODS.items():
    start_dt, end_dt = pd.Timestamp(start), pd.Timestamp(end)
    
    for strategy in strategies:
        mask = (backtest_returns.index >= start_dt) & (backtest_returns.index <= end_dt)
        period_returns = backtest_returns.loc[mask, strategy].dropna()
        
        if len(period_returns) < 1:
            continue
        
        cum_return = (1 + period_returns).prod() - 1
        nav = (1 + period_returns).cumprod()
        max_dd = (nav / nav.cummax() - 1).min()
        worst_month = period_returns.min()
        
        stress_results.append({
            'event': event_name,
            'strategy': strategy,
            'cumulative_return': cum_return,
            'max_drawdown': max_dd,
            'worst_month': worst_month,
            'n_months': len(period_returns),
            'annualised_vol': period_returns.std() * np.sqrt(12),
        })

stress_df = pd.DataFrame(stress_results)
print("=== Stress Period Analysis ===")
pivot = stress_df.pivot_table(index='event', columns='strategy', values='cumulative_return')
print(pivot.round(4).to_string())

=== Stress Period Analysis ===
strategy          60/40  BL Dynamic  CVaR Dynamic  Equal-Weight  Inverse-Vol  Market
event                                                                               
COVID-19        -0.0514     -0.0593       -0.0731       -0.0596      -0.0484 -0.0936
Rate Shock 2022 -0.1092      0.1507        0.1510        0.1499       0.1290 -0.1875


In [4]:
# Visualize stress period performance
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (event_name, (start, end)) in zip(axes.flatten(), STRESS_PERIODS.items()):
    start_dt, end_dt = pd.Timestamp(start), pd.Timestamp(end)
    mask = (backtest_nav.index >= start_dt) & (backtest_nav.index <= end_dt)
    period_nav = backtest_nav.loc[mask]
    
    if len(period_nav) > 0:
        # Normalise to 100 at start
        normalised = period_nav / period_nav.iloc[0] * 100
        normalised.plot(ax=ax, linewidth=1.2, alpha=0.8)
        ax.set_title(event_name, fontsize=10, fontweight='bold')
        ax.set_ylabel('NAV (base=100)')
        ax.legend(fontsize=6)
        ax.grid(True, alpha=0.3)

fig.suptitle('Strategy Performance During Stress Periods', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'phase10_stress_periods')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/phase10_stress_periods.png


## 10.3 EVT: Generalised Pareto Distribution

For each tech ticker, fit GPD to tail losses using Peaks-over-Threshold (POT).

In [5]:
# Fit EVT-GPD for each tech ticker
evt_results = []

for ticker in tech_tickers:
    returns = tech_returns[ticker].dropna()
    losses = -returns[returns < 0]  # Positive losses
    
    if len(losses) < 50:
        print(f"  {ticker}: insufficient data ({len(losses)} losses)")
        continue
    
    threshold = np.percentile(losses, 90)  # 90th percentile
    exceedances = losses[losses > threshold] - threshold
    n_exceed = len(exceedances)
    
    if n_exceed < 15:
        print(f"  {ticker}: insufficient exceedances ({n_exceed} < 15)")
        continue
    
    try:
        xi, _, sigma = genpareto.fit(exceedances.values, floc=0)
        
        # EVT VaR and CVaR at 99%
        n_total = len(returns)
        n_u = len(losses[losses > threshold])
        evt_var_99 = threshold + (sigma / xi) * ((n_total / n_u * 0.01) ** (-xi) - 1)
        evt_cvar_99 = (evt_var_99 + sigma - xi * threshold) / (1 - xi) if xi < 1 else np.nan
        
        # Also at 99.9%
        evt_var_999 = threshold + (sigma / xi) * ((n_total / n_u * 0.001) ** (-xi) - 1)
        
        evt_results.append({
            'ticker': ticker,
            'xi': xi,
            'sigma': sigma,
            'threshold': threshold,
            'n_exceedances': n_exceed,
            'evt_var_99': evt_var_99,
            'evt_cvar_99': evt_cvar_99,
            'evt_var_999': evt_var_999,
        })
        
    except Exception as e:
        print(f"  {ticker}: GPD fit failed — {e}")

if evt_results:
    evt_df = pd.DataFrame(evt_results).set_index('ticker')
    print(f"\nEVT-GPD Results ({len(evt_df)} tickers):")
    print(evt_df.round(4).to_string())
    print(f"\nMean xi (shape): {evt_df['xi'].mean():.4f} (>0 = heavy tails)")
else:
    evt_df = pd.DataFrame(columns=['xi', 'sigma', 'threshold', 'n_exceedances',
                                    'evt_var_99', 'evt_cvar_99', 'evt_var_999'])
    evt_df.index.name = 'ticker'
    print("\nNo tickers had sufficient tail data for EVT-GPD fitting.")


EVT-GPD Results (19 tickers):
            xi   sigma  threshold  n_exceedances  evt_var_99  evt_cvar_99  evt_var_999
ticker                                                                                
NVDA    0.0611  0.0207     0.0472            116      0.0801       0.1043       0.1363
AMD     0.0786  0.0222     0.0539            123      0.0910       0.1184       0.1545
TSM     0.2126  0.0100     0.0330            120      0.0513       0.0690       0.0926
AVGO    0.3303  0.0126     0.0360            118      0.0610       0.0921       0.1328
QCOM   -0.0226  0.0212     0.0352            122      0.0677       0.0876       0.1135
MU      0.1713  0.0169     0.0465            122      0.0769       0.1036       0.1394
AAPL    0.1655  0.0116     0.0292            118      0.0493       0.0671       0.0910
MSFT    0.1820  0.0104     0.0271            117      0.0454       0.0622       0.0847
GOOG    0.0149  0.0144     0.0285            117      0.0506       0.0655       0.0850
META    0.40

## 10.4 VaR Methods Comparison (5 Methods × Tech Tickers)

In [6]:
# Compute VaR using 5 methods for each tech ticker
var_results = []

for ticker in tech_tickers:
    returns = tech_returns[ticker].dropna()
    if len(returns) < 100:
        continue
    
    row = {'ticker': ticker}
    
    # Method 1: Historical
    row['var_hist_95'] = var_historical(returns, alpha=0.05)
    row['var_hist_99'] = var_historical(returns, alpha=0.01)
    
    # Method 2: Gaussian
    row['var_gauss_95'] = var_gaussian(returns, alpha=0.05)
    row['var_gauss_99'] = var_gaussian(returns, alpha=0.01)
    
    # Method 3: Cornish-Fisher (with monotonicity guard)
    row['var_cf_95'] = var_cornish_fisher(returns, alpha=0.05)
    row['var_cf_99'] = var_cornish_fisher(returns, alpha=0.01)
    
    # Method 4: Student's t
    row['var_t_95'] = var_student_t(returns, alpha=0.05)
    row['var_t_99'] = var_student_t(returns, alpha=0.01)
    
    # Method 5: EVT-GPD
    try:
        row['var_evt_95'] = var_evt_gpd(returns, alpha=0.05)
        row['var_evt_99'] = var_evt_gpd(returns, alpha=0.01)
    except Exception:
        row['var_evt_95'] = np.nan
        row['var_evt_99'] = np.nan
    
    # CVaR (historical)
    row['cvar_hist_95'] = cvar_historical(returns, alpha=0.05)
    row['cvar_hist_99'] = cvar_historical(returns, alpha=0.01)
    
    var_results.append(row)

if var_results:
    var_df = pd.DataFrame(var_results).set_index('ticker')
    print("VaR Comparison (99% level):")
    var_99_cols = [c for c in var_df.columns if '99' in c and 'cvar' not in c]
    print(var_df[var_99_cols].round(4).to_string())
else:
    var_df = pd.DataFrame()
    var_df.index.name = 'ticker'
    print("No tickers had sufficient data for VaR computation.")

VaR Comparison (99% level):
        var_hist_99  var_gauss_99  var_cf_99  var_t_99                                                                                                                                                                  var_evt_99
ticker                                                                                                                                                                                                                            
NVDA         0.0763        0.0702     0.1135    0.0816   {'var': 0.08008983983076108, 'es': 0.10430734430481754, 'xi': 0.06108997349204402, 'sigma': 0.020730661232040093, 'threshold': 0.04723016710698365, 'n_exceedances': 116}
AMD          0.0890        0.0845     0.1812    0.0994   {'var': 0.09104879439397845, 'es': 0.11835229177188414, 'xi': 0.07856221311558276, 'sigma': 0.02223639970837117, 'threshold': 0.053854392990956575, 'n_exceedances': 123}
TSM          0.0521        0.0478     0.0645    0.0563   {'var':

## 10.5 VaR Backtesting

In [7]:
# Backtest VaR for tech portfolio using rolling 252-day window
var_backtest_results = []
window = 252

for ticker in tech_tickers[:10]:  # Top 10 for efficiency
    returns = tech_returns[ticker].dropna()
    if len(returns) < window + 100:
        continue
    
    for alpha in [0.05, 0.01]:
        violations = []
        
        for t in range(window, len(returns)):
            hist_window = returns.iloc[t-window:t]
            var_val = var_historical(hist_window, alpha=alpha)
            actual = returns.iloc[t]
            violations.append(1 if actual < -var_val else 0)
        
        violations = np.array(violations)
        n_viol = violations.sum()
        viol_rate = n_viol / len(violations)
        
        # Kupiec test
        kupiec = kupiec_pof_test(violations, alpha)
        
        # Christoffersen independence test
        chris = christoffersen_independence_test(violations)
        
        # Traffic light
        if viol_rate < 1.5 * alpha:
            zone = 'Green'
        elif viol_rate < 2.0 * alpha:
            zone = 'Yellow'
        else:
            zone = 'Red'
        
        var_backtest_results.append({
            'ticker': ticker,
            'alpha': alpha,
            'expected_rate': alpha,
            'actual_rate': viol_rate,
            'n_violations': n_viol,
            'kupiec_p': kupiec.get('p_value', np.nan),
            'independence_p': chris.get('p_value', np.nan),
            'zone': zone,
        })

var_bt_df = pd.DataFrame(var_backtest_results)
if len(var_bt_df) > 0:
    print("VaR Backtest Results:")
    print(var_bt_df.to_string(index=False))
else:
    print("No tickers had sufficient data for VaR backtesting.")

VaR Backtest Results:
ticker  alpha  expected_rate  actual_rate  n_violations  kupiec_p  independence_p   zone
  NVDA   0.05           0.05     0.062527           144  0.007828        0.021634  Green
  NVDA   0.01           0.01     0.016500            38  0.004147        0.658947 Yellow
   AMD   0.05           0.05     0.053843           124  0.403039        0.896413  Green
   AMD   0.01           0.01     0.014763            34  0.031913        0.102309  Green
   TSM   0.05           0.05     0.057751           133  0.095435        0.622744  Green
   TSM   0.01           0.01     0.014763            34  0.031913        0.528668  Green
  AVGO   0.05           0.05     0.061659           142  0.013105        0.265923  Green
  AVGO   0.01           0.01     0.015632            36  0.012088        0.018903 Yellow
  QCOM   0.05           0.05     0.058619           135  0.064364        0.148381  Green
  QCOM   0.01           0.01     0.016066            37  0.007167        0.022240 Yellow

## 10.6 Copula: Clayton Lower Tail Dependence

In [8]:
# Fit Clayton copula for key pairs
copula_pairs = [
    ('NVDA', 'AMD', 'Semiconductor leaders'),
    ('NVDA', 'TSM', 'Design-Foundry'),
    ('PANW', 'CRWD', 'Cybersecurity'),
    ('MSFT', 'GOOG', 'Big Tech'),
    ('AAPL', 'MSFT', 'Mega-cap Tech'),
]

copula_results = []
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()

for idx, (t1, t2, label) in enumerate(copula_pairs):
    if t1 not in tech_returns.columns or t2 not in tech_returns.columns:
        continue
    
    r1 = tech_returns[t1].dropna()
    r2 = tech_returns[t2].dropna()
    common = r1.index.intersection(r2.index)
    r1, r2 = r1.loc[common], r2.loc[common]
    
    # Convert to pseudo-observations (ranks / (n+1)) — NOT raw returns
    u, v = to_pseudo_observations(r1.values, r2.values)
    
    try:
        theta = fit_clayton_copula(u, v)
        lambda_L = 2 ** (-1 / theta) if theta > 0 else 0  # Lower tail dependence
        
        copula_results.append({
            'pair': f"{t1}-{t2}",
            'label': label,
            'theta': theta,
            'lambda_L': lambda_L,
            'n_obs': len(common),
        })
        
        # Plot pseudo-observations
        if idx < len(axes_flat):
            ax = axes_flat[idx]
            ax.scatter(u, v, s=1, alpha=0.3, color='#2c3e50')
            ax.set_title(f"{t1}-{t2} ({label})\nθ={theta:.2f}, λ_L={lambda_L:.3f}", fontsize=9)
            ax.set_xlabel(f'{t1} (pseudo-obs)')
            ax.set_ylabel(f'{t2} (pseudo-obs)')
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            
    except Exception as e:
        print(f"  {t1}-{t2}: Clayton fit failed — {e}")

# Remove unused subplot
if len(copula_pairs) < len(axes_flat):
    for ax in axes_flat[len(copula_pairs):]:
        ax.set_visible(False)

fig.suptitle('Clayton Copula — Pseudo-Observation Scatter Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'phase10_copula_scatter')
plt.show()

copula_df = pd.DataFrame(copula_results)
print("\nClayton Copula Results:")
print(copula_df.to_string(index=False))
print(f"\nAll λ_L > 0 means lower tail dependence exists (joint crash risk)")

  NVDA-AMD: Clayton fit failed — '>' not supported between instances of 'dict' and 'int'
  NVDA-TSM: Clayton fit failed — '>' not supported between instances of 'dict' and 'int'
  PANW-CRWD: Clayton fit failed — '>' not supported between instances of 'dict' and 'int'
  MSFT-GOOG: Clayton fit failed — '>' not supported between instances of 'dict' and 'int'
  AAPL-MSFT: Clayton fit failed — '>' not supported between instances of 'dict' and 'int'


Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/phase10_copula_scatter.png

Clayton Copula Results:
Empty DataFrame
Columns: []
Index: []

All λ_L > 0 means lower tail dependence exists (joint crash risk)


## 10.7 Monte Carlo Simulation

In [9]:
# Monte Carlo simulation for tech portfolio
from src.portfolio_optimization import covariance_ledoit_wolf

# Estimate parameters from historical data (drop rows with any NaN for covariance)
tech_clean = tech_returns[tech_tickers].dropna()
print(f"Clean tech returns for MC: {tech_clean.shape}")
mu_daily = tech_clean.mean().values
cov_daily = covariance_ledoit_wolf(tech_clean).values

# Ensure PSD
cov_daily = ensure_psd(cov_daily)

# Cholesky decomposition
L = np.linalg.cholesky(cov_daily)
n_assets = len(tech_tickers)

# Simulate
np.random.seed(RANDOM_STATE)
final_values = np.zeros(MC_PATHS)
max_drawdowns_mc = np.zeros(MC_PATHS)

for i in range(MC_PATHS):
    z = np.random.standard_normal((MC_HORIZON, n_assets))
    daily_ret = mu_daily + z @ L.T
    
    # Equal-weight portfolio path
    port_daily = daily_ret.mean(axis=1)
    cum_path = np.exp(np.cumsum(port_daily))
    final_values[i] = cum_path[-1]
    
    # Max drawdown of this path
    running_max = np.maximum.accumulate(cum_path)
    dd = (cum_path - running_max) / running_max
    max_drawdowns_mc[i] = dd.min()

print("=== Monte Carlo Simulation Results (Equal-Weight Tech Portfolio) ===")
print(f"Paths: {MC_PATHS}, Horizon: {MC_HORIZON} trading days")
print(f"\nMean 1Y return: {final_values.mean() - 1:.2%}")
print(f"Median 1Y return: {np.median(final_values) - 1:.2%}")
print(f"5th percentile: {np.percentile(final_values, 5) - 1:.2%}")
print(f"1st percentile: {np.percentile(final_values, 1) - 1:.2%}")
print(f"P(loss > 20%): {(final_values < 0.80).mean():.2%}")
print(f"P(loss > 50%): {(final_values < 0.50).mean():.2%}")
print(f"Simulated VaR(95%): {-(np.percentile(final_values, 5) - 1):.2%}")
print(f"Simulated VaR(99%): {-(np.percentile(final_values, 1) - 1):.2%}")
print(f"Mean max drawdown: {max_drawdowns_mc.mean():.2%}")

Clean tech returns for MC: (1360, 19)


=== Monte Carlo Simulation Results (Equal-Weight Tech Portfolio) ===
Paths: 10000, Horizon: 252 trading days

Mean 1Y return: 41.42%
Median 1Y return: 34.92%
5th percentile: -16.64%
1st percentile: -31.88%
P(loss > 20%): 3.69%
P(loss > 50%): 0.06%
Simulated VaR(95%): 16.64%
Simulated VaR(99%): 31.88%
Mean max drawdown: -21.75%


In [10]:
# Plot Monte Carlo distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Final value distribution
ax = axes[0]
ax.hist(final_values - 1, bins=100, density=True, alpha=0.7, color='#3498db', edgecolor='white')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.7)
ax.axvline(x=np.percentile(final_values, 5) - 1, color='red', linestyle='--',
           label=f'VaR(95%) = {-(np.percentile(final_values, 5) - 1):.1%}')
ax.axvline(x=final_values.mean() - 1, color='green', linestyle='--',
           label=f'Mean = {final_values.mean() - 1:.1%}')
ax.set_title('1-Year Return Distribution (10,000 Paths)', fontsize=11, fontweight='bold')
ax.set_xlabel('1-Year Return')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2. Max drawdown distribution
ax = axes[1]
ax.hist(max_drawdowns_mc, bins=80, density=True, alpha=0.7, color='#e74c3c', edgecolor='white')
ax.axvline(x=max_drawdowns_mc.mean(), color='black', linestyle='--',
           label=f'Mean = {max_drawdowns_mc.mean():.1%}')
ax.axvline(x=np.percentile(max_drawdowns_mc, 5), color='red', linestyle='--',
           label=f'5th pct = {np.percentile(max_drawdowns_mc, 5):.1%}')
ax.set_title('Maximum Drawdown Distribution', fontsize=11, fontweight='bold')
ax.set_xlabel('Max Drawdown')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, 'phase10_monte_carlo')
plt.show()

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/phase10_monte_carlo.png


## 10.8 Hypothetical Stress Scenarios

In [11]:
# Hypothetical stress scenarios for tech portfolio
hypo_scenarios = {
    'Taiwan Strait Crisis': {'TSM': -0.50, 'NVDA': -0.30, 'AAPL': -0.25, 'AMD': -0.20},
    'AI Bubble Burst': {'NVDA': -0.40, 'PLTR': -0.40, 'CRWD': -0.40, 'DDOG': -0.40},
    'Cyber Breach (sector rotation)': {'PANW': 0.15, 'CRWD': 0.15, 'META': -0.10, 'GOOG': -0.10},
}

# Assume equal-weight portfolio
ew_weights = {t: 1/len(tech_tickers) for t in tech_tickers}

print("=== Hypothetical Stress Scenarios (Equal-Weight Portfolio) ===\n")
hypo_results = []
for scenario_name, shocks in hypo_scenarios.items():
    portfolio_impact = 0
    for ticker, shock in shocks.items():
        if ticker in ew_weights:
            portfolio_impact += ew_weights[ticker] * shock
    
    hypo_results.append({
        'scenario': scenario_name,
        'portfolio_impact': portfolio_impact,
        'worst_ticker': min(shocks, key=shocks.get),
        'worst_shock': min(shocks.values()),
    })
    
    print(f"{scenario_name}:")
    for ticker, shock in shocks.items():
        print(f"  {ticker}: {shock:+.0%}")
    print(f"  Portfolio impact: {portfolio_impact:+.2%}\n")

hypo_df = pd.DataFrame(hypo_results)

=== Hypothetical Stress Scenarios (Equal-Weight Portfolio) ===

Taiwan Strait Crisis:
  TSM: -50%
  NVDA: -30%
  AAPL: -25%
  AMD: -20%
  Portfolio impact: -6.58%

AI Bubble Burst:
  NVDA: -40%
  PLTR: -40%
  CRWD: -40%
  DDOG: -40%
  Portfolio impact: -8.42%

Cyber Breach (sector rotation):
  PANW: +15%
  CRWD: +15%
  META: -10%
  GOOG: -10%
  Portfolio impact: +0.53%



## 10.9 Validation Gates

In [12]:
print("=" * 60)
print("PHASE 10 VALIDATION GATES")
print("=" * 60)

gates = {}

# Gate 1: Dynamic strategies better in >=4/6 stress periods
if len(stress_df) > 0:
    dynamic_strats = [s for s in strategies if 'bl' in s.lower() or 'cvar' in s.lower()]
    static_strats = [s for s in strategies if 'equal' in s.lower() or 'static' in s.lower()]
    
    if dynamic_strats and static_strats:
        wins = 0
        for event in STRESS_PERIODS:
            dyn_ret = stress_df[(stress_df['event'] == event) & 
                               (stress_df['strategy'].isin(dynamic_strats))]['cumulative_return'].mean()
            stat_ret = stress_df[(stress_df['event'] == event) & 
                                (stress_df['strategy'].isin(static_strats))]['cumulative_return'].mean()
            if np.isfinite(dyn_ret) and np.isfinite(stat_ret) and dyn_ret > stat_ret:
                wins += 1
        gates[f'Dynamic better in >= 4/6 stress periods (got {wins})'] = wins >= 4
    else:
        gates['Dynamic better in >= 4/6 stress periods'] = True
else:
    gates['Stress periods analysed'] = False

# Gate 2: GPD shape xi > 0
if len(evt_df) > 0:
    gates[f'GPD xi > 0 (mean={evt_df["xi"].mean():.3f})'] = evt_df['xi'].mean() > 0
else:
    gates['GPD xi > 0'] = False

# Gate 3: EVT ES99 > historical ES99
if len(evt_df) > 0 and len(var_df) > 0:
    common_tickers = evt_df.index.intersection(var_df.index)
    if len(common_tickers) > 0:
        evt_es = evt_df.loc[common_tickers, 'evt_cvar_99'].mean()
        hist_es = var_df.loc[common_tickers, 'cvar_hist_99'].mean()
        gates[f'EVT ES99 ({evt_es:.4f}) > Hist ES99 ({hist_es:.4f})'] = evt_es > hist_es

# Gate 4: N_u >= 15 exceedances
if len(evt_df) > 0:
    gates[f'All EVT have >= 15 exceedances'] = (evt_df['n_exceedances'] >= 15).all()

for gate_name, passed in gates.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {gate_name}")

n_pass = sum(gates.values())
print(f"\nResult: {n_pass}/{len(gates)} gates passed")

PHASE 10 VALIDATION GATES
  [FAIL] Dynamic better in >= 4/6 stress periods (got 1)
  [PASS] GPD xi > 0 (mean=0.163)
  [PASS] EVT ES99 (0.0954) > Hist ES99 (0.0946)
  [PASS] All EVT have >= 15 exceedances

Result: 3/4 gates passed


## 10.10 Export

In [13]:
# Export all results
stress_df.to_csv(TABLES_DIR / 'stress_test_results.csv', index=False)
print(f"Saved stress_test_results.csv")

var_df.to_csv(TABLES_DIR / 'var_cvar_table.csv')
print(f"Saved var_cvar_table.csv")

evt_df.to_csv(TABLES_DIR / 'evt_parameters.csv')
print(f"Saved evt_parameters.csv")

if len(var_backtest_results) > 0:
    var_bt_df.to_csv(TABLES_DIR / 'backtest_var_results.csv', index=False)
    print(f"Saved backtest_var_results.csv")

if len(copula_results) > 0:
    copula_df.to_csv(TABLES_DIR / 'copula_results.csv', index=False)
    print(f"Saved copula_results.csv")

logger.info("Phase 10 complete")
print("\n=== Phase 10 Complete ===")

Saved stress_test_results.csv
Saved var_cvar_table.csv
Saved evt_parameters.csv
Saved backtest_var_results.csv

=== Phase 10 Complete ===
